In [1]:
import sys
!{sys.executable} -m pip install -q sentencepiece protobuf


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install -q transformers datasets

import pandas as pd, numpy as np, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

train = pd.read_csv('C:\\Users\\SAGAR\\OneDrive\\Desktop\\dl_genai\\dl-genai-project\\data\\train.csv')
test  = pd.read_csv('C:\\Users\\SAGAR\\OneDrive\\Desktop\\dl_genai\\dl-genai-project\\data\\test.csv')
OPTS = ['A','B','C','D','E']
train['label'] = train['answer'].map({o:i for i,o in enumerate(OPTS)})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def make_text(row):
    """prompt + saare options ek input string mein (seq-classification format)"""
    return (str(row['prompt']) + " | A: " + str(row['A']) + " | B: " + str(row['B'])
            + " | C: " + str(row['C']) + " | D: " + str(row['D']) + " | E: " + str(row['E']))

train['text'] = train.apply(make_text, axis=1)
test['text']  = test.apply(make_text, axis=1)

def finetune(model_name, out_dir):
    tk = AutoTokenizer.from_pretrained(model_name)
    def tok_fn(ex): return tk(ex['text'], truncation=True, max_length=256)
    ds = Dataset.from_pandas(train[['text','label']]).map(tok_fn, remove_columns=['text'])
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
    args = TrainingArguments(output_dir=out_dir, num_train_epochs=2,
                             per_device_train_batch_size=8, learning_rate=2e-5,
                             fp16=False, report_to=[], save_strategy="no",
                             logging_steps=50, seed=42)
    tr = Trainer(model=model, args=args, train_dataset=ds,
                 processing_class=tk)
    tr.train()
    return model.to(DEVICE).eval(), tk

print("Fine-tuning DeBERTa-v3-small..."); deb_model, deb_tok = finetune("microsoft/deberta-v3-small", "deb_ft")
print("Fine-tuning RoBERTa-base...");     rob_model, rob_tok = finetune("roberta-base", "rob_ft")
print("Setup done.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Fine-tuning DeBERTa-v3-small...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.de

In [ ]:
def get_probs(model, tk, texts, batch_size=16):
    all_p = []
    for i in range(0, len(texts), batch_size):
        enc = tk(texts[i:i+batch_size], truncation=True, max_length=256,
                 padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        all_p.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(all_p)

In [ ]:
t25 = [train.iloc[25]['text']]
p_deb = get_probs(deb_model, deb_tok, t25)[0]
p_rob = get_probs(rob_model, rob_tok, t25)[0]

i = int(p_deb.argmax())
print(f"ANSWER Q1: {OPTS[i]}, {round(float(p_deb[i]), 4)}")
print("(RoBERTa top for reference:", OPTS[int(p_rob.argmax())], round(float(p_rob.max()),4), ")")

In [ ]:
p_avg = (p_deb + p_rob) / 2
print("ANSWER Q2:", OPTS[int(p_avg.argmax())])

p_wt = 0.7 * p_deb + 0.3 * p_rob
print("ANSWER Q3:", OPTS[int(p_wt.argmax())])

In [ ]:
order = np.argsort(-p_wt)[:3]
print("ANSWER Q4:", " ".join(OPTS[j] for j in order))

In [ ]:
deb_test = get_probs(deb_model, deb_tok, test['text'].tolist())
rob_test = get_probs(rob_model, rob_tok, test['text'].tolist())
ens_test = 0.7 * deb_test + 0.3 * rob_test

top3 = [" ".join(OPTS[j] for j in np.argsort(-row)[:3]) for row in ens_test]
sub = pd.DataFrame({"id": test['id'], "prediction": top3})
sub.to_csv("submission.csv", index=False)
print("ANSWER Q5:", len(sub))     # expect 500

In [ ]:
texts_orig = test['text'].head(50).tolist()
texts_aug  = ["Answer the following multiple-choice question carefully: " + t for t in texts_orig]

p_orig = get_probs(deb_model, deb_tok, texts_orig)
p_aug  = get_probs(deb_model, deb_tok, texts_aug)
p_tta  = (p_orig + p_aug) / 2

changed = int((p_orig.argmax(1) != p_tta.argmax(1)).sum())
print("ANSWER Q6:", changed)

In [ ]:
d100 = deb_test[:100]; e100 = ens_test[:100]

# Q7: different top-1
print("ANSWER Q7:", int((d100.argmax(1) != e100.argmax(1)).sum()))

# Q8: positive confidence gain
gain = e100.max(1) - d100.max(1)
print("ANSWER Q8:", int((gain > 0).sum()))

# Q9: any change in ordered top-3
def top3_str(p): return " ".join(OPTS[j] for j in np.argsort(-p)[:3])
diff3 = sum(top3_str(d100[i]) != top3_str(e100[i]) for i in range(100))
print("ANSWER Q9:", diff3)

In [ ]:
# test.csv mein labels nahi hain, isliye "validation samples" = train ke first 100 rows
val100 = train.head(100)
d_val = get_probs(deb_model, deb_tok, val100['text'].tolist())
r_val = get_probs(rob_model, rob_tok, val100['text'].tolist())
e_val = 0.7 * d_val + 0.3 * r_val

total = 0.0
for i in range(100):
    gt = val100.iloc[i]['answer']
    top3_letters = [OPTS[j] for j in np.argsort(-e_val[i])[:3]]
    for r, p in enumerate(top3_letters):
        if p == gt: total += 1.0/(r+1); break
print("ANSWER Q10:", round(total/100, 4))